# Chapter 01 · Mathematics: Inputs, predictions, loss, and gradients

[Chapter guide](README.md) · [Problem-to-mathematics guide](../MATHEMATICS.md) · [Course index](../README.md)

**Status: optional reference for later; not yet reviewed.** The programming lessons are already covered. Continue the programming-led mentorship without needing to complete this companion first. Use small numerical explanations when helpful, and revisit formal notation gradually.

Read one section at a time: words → a small example → symbols → code → your explanation. Symbols are shorthand for ideas you already used. You do not need to memorize all the formulas before continuing.

**Prerequisites:** Addition, multiplication, and averaging; we explain squares and slopes as they appear.

The runnable examples use only Python and the installed PyTorch. Start in a fresh kernel and run Setup, then proceed in order. Each example keeps its calculations small and supplies its own data. Code cells are saved unexecuted so you can make your own predictions first.

## Contents

1. [1. Translate a problem into a prediction](#prediction)
2. [2. Turn errors into one loss](#loss)
3. [3. Read gradients as local slopes](#gradients)
4. [4. Turn gradients into an update](#update)

## Setup

In [ ]:
import torch

<a id="prediction"></a>
## 1. Translate a problem into a prediction


**Problem in words:** given an input number, predict the corresponding output number. Think of our made-up data as delivery distance in kilometres and delivery time in minutes: distances `[0, 1, 2, 3]` have recorded times `[2, 5, 8, 11]`. This is a toy example, not a claim about real deliveries.

Choose the information known when predicting as the **input**, and the quantity we want to predict as the **target**. Choosing a straight-line model is an assumption about the relationship; mathematics does not choose that assumption for us.

| Symbol | Read it as | Role in this example |
|---|---|---|
| $x$ | input | One distance |
| $y$ | target | Its recorded time |
| $\hat y$ | y-hat | Our predicted time; the hat distinguishes a prediction from a target |
| $w$ | weight | How much the prediction changes per unit increase in input |
| $b$ | bias | The prediction when input is zero |

Our proposed prediction rule is:

$$\hat y = wx+b$$

Read that as **multiply the input by the weight, then add the bias**. A function is a rule mapping an input to an output. We can also write this rule as $f(x)=wx+b$; $f(x)$ means the output produced by the rule for input $x$.

For $x=2$, $w=1$, and $b=0$, the prediction is $1\times2+0=2$. The target is still $8$. The supplied target and the current model prediction are different quantities. The weights and bias are shared across examples; inputs and targets vary by example.

**Connection:** programming Lessons 03–04 in [01_lessons.ipynb](01_lessons.ipynb).

**Before running:** what does $\hat y$ become if the weight is `2` and bias is `0.5`, keeping input `2`?


**My prediction or explanation:** _write here_

In [ ]:
# Translate the formula literally: prediction = weight * input + bias.
x_example, y_example = 2.0, 8.0
weight, bias = 2.0, 0.5
prediction = weight * x_example + bias
print("Input:", x_example)
print("Target:", y_example)
print("Prediction:", prediction)
print("Signed error (prediction - target):", prediction - y_example)

<a id="loss"></a>
## 2. Turn errors into one loss


An **error** compares one prediction with its target. We use $e=\hat y-y$: negative means the prediction is too small, positive means too large.

Squaring means multiplying a number by itself. Both $(-2)^2$ and $2^2$ equal $4$. Squared errors cannot cancel each other, and a larger miss receives a larger penalty. The units of a squared error are squared units; they are not the original units of the prediction.

At weight `1` and bias `0`, our four predictions are `[0, 1, 2, 3]`. Their errors are `[-2, -4, -6, -8]`, and their squared errors are `[4, 16, 36, 64]`.

The **mean squared error** is their average:

$$L=\frac{4+16+36+64}{4}=30$$

Now compress that same calculation into notation:

$$L(w,b)=\frac{1}{n}\sum_{i=1}^{n}(wx_i+b-y_i)^2$$

| New symbol | Meaning |
|---|---|
| $L$ | Loss: one number measuring the current errors |
| $n$ | Number of examples being averaged |
| $i$ | Example index, from 1 to $n$ in this formula; Python indexes start at 0 |
| $x_i,y_i$ | Input and target for example $i$ |
| $\sum$ | Sum: add the expression once for each example |

For our one-output examples, `((predictions - targets) ** 2).mean()` implements this formula. For multiple outputs, PyTorch's default mean averages all error elements, not just rows. See [MSELoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html).

MSE is a choice suited to this numeric prediction exercise; other tasks may need different losses. Zero MSE means every prediction in the averaged data matches its target. It does not by itself describe new, unseen data.

**Connection:** programming Lesson 05. **Before running:** calculate the four squared errors and their average by hand.


**My prediction or explanation:** _write here_

In [ ]:
# Recreate the exact four-example arithmetic behind loss = 30.
x_values = torch.tensor([0., 1., 2., 3.])
y_values = torch.tensor([2., 5., 8., 11.])
predictions = 1.0 * x_values + 0.0
errors = predictions - y_values
squared_errors = errors ** 2
print("Errors:", errors.tolist())
print("Squared errors:", squared_errors.tolist())
print("Mean squared loss:", squared_errors.mean().item())

<a id="gradients"></a>
## 3. Read gradients as local slopes


A **slope** describes how one quantity changes when another changes. A gradient tells us the local slope of the loss with respect to each parameter, at the current parameter values.

Hold bias at zero and change only the weight slightly:

| Weight | Full training loss |
|---:|---:|
| 0.99 | 30.20035 |
| 1.00 | 30.00000 |
| 1.01 | 29.80035 |

Increasing the weight by `0.01` lowers loss by about `0.2`. Dividing the loss change by the weight change gives a local slope near `-20`. The derivative notation below names this idea precisely.

Imagine changing just the weight a tiny amount while holding the bias fixed. The derivative $\partial L/\partial w$ asks: **how quickly would loss change per unit change in weight, nearby?** The partial symbol $\partial$ means we change this one parameter while holding the others fixed. $\partial L/\partial b$ asks the same question for bias.

For a tiny change $\Delta w$ (delta means a change), the local approximation is:

$$\Delta L\approx\frac{\partial L}{\partial w}\Delta w$$

The approximately-equal sign matters: a large parameter change need not follow this local approximation. A negative gradient suggests that a sufficiently small increase in that parameter lowers loss locally. A positive gradient suggests a small decrease. A zero gradient does not always mean all errors are zero.

For one example, loss is the squared error $e^2$. Its slope with respect to error is $2e$. Changing the bias changes prediction at rate `1`; changing the weight changes prediction at rate $x$. Multiplying these rates is the idea behind the **chain rule**:

$$g_b=2e,\qquad g_w=2ex$$

We do not need a complete calculus course to use this chain of changes: **parameter → prediction → error → squared error**.

For mean loss over $n$ examples, average the contributions:

$$g_w=\frac{2}{n}\sum_{i=1}^{n}e_ix_i,\qquad g_b=\frac{2}{n}\sum_{i=1}^{n}e_i$$

At weights `1`, bias `0`, our errors are `[-2, -4, -6, -8]`. Weight-gradient contributions are `[0, -8, -24, -48]`, averaging to `-20`. Bias-gradient contributions are `[-4, -8, -12, -16]`, averaging to `-10`.

**Connection:** programming Lessons 06 and 09. **Before running:** why is the weight contribution zero for input `0`, while the bias contribution can be nonzero?


**My prediction or explanation:** _write here_

In [ ]:
# Compare the explicit derivative formulas with PyTorch autograd.
x_values = torch.tensor([0., 1., 2., 3.])
y_values = torch.tensor([2., 5., 8., 11.])
weight = torch.tensor(1., requires_grad=True)
bias = torch.tensor(0., requires_grad=True)
errors = weight * x_values + bias - y_values
loss = (errors ** 2).mean()
formula_weight_gradient = (2 * errors.detach() * x_values).mean()
formula_bias_gradient = (2 * errors.detach()).mean()
loss.backward()
print("Formula weight gradient:", formula_weight_gradient.item())
print("Autograd weight gradient:", weight.grad.item())
print("Formula bias gradient:", formula_bias_gradient.item())
print("Autograd bias gradient:", bias.grad.item())

<a id="update"></a>
## 4. Turn gradients into an update


**Optimization** means searching for parameter values that reduce the chosen loss. SGD takes a step opposite the current gradient:

$$w_{\mathrm{new}}=w_{\mathrm{old}}-\eta g_w,\qquad b_{\mathrm{new}}=b_{\mathrm{old}}-\eta g_b$$

The Greek letter $\eta$ (eta) is our learning rate: how much we scale the gradient before changing a parameter. The gradient is not a new weight; it is information used to calculate the update.

For $\eta=0.05$, $w=1$, $b=0$, $g_w=-20$, and $g_b=-10$:

$$w_{\mathrm{new}}=1-0.05(-20)=2,\qquad b_{\mathrm{new}}=0-0.05(-10)=0.5$$

Both values increase because we subtract negative numbers. Recompute predictions and loss afterward, then recompute gradients before the next update. The loss is a function of the current parameters, so its slopes generally change as parameters change.

A step that is too large can overshoot. More updates do not imply that loss must eventually rise, nor that it must reach exactly zero. The model, data, optimizer, and step size determine the behavior.

**Connection:** programming Lessons 07–11 and [SGD](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html).

**Before running:** use the new weight and bias to predict for input `2`.


**My prediction or explanation:** _write here_

In [ ]:
# Apply the known gradient values and measure the new full-batch loss.
learning_rate = 0.05
new_weight = 1.0 - learning_rate * (-20.0)
new_bias = 0.0 - learning_rate * (-10.0)
x_values = torch.tensor([0., 1., 2., 3.])
y_values = torch.tensor([2., 5., 8., 11.])
new_predictions = new_weight * x_values + new_bias
new_loss = ((new_predictions - y_values) ** 2).mean()
print("New weight:", new_weight)
print("New bias:", new_bias)
print("New predictions:", new_predictions.tolist())
print("Loss after update:", new_loss.item())

**My observation:** _write here_

Explain the full chain in your own words: input and target → chosen prediction rule → error → loss → local slopes → parameter update. Choosing the model and loss is part of translating a problem into mathematics.